# Phase 0.5 - Environment Setup and Data Verification

Run all cells from top to bottom in Google Colab after mounting Drive. This notebook downloads the raw archives, extracts them, verifies HDF5 structure, saves sanity plots, and writes `results/00_environment_report.json`.


In [3]:
from google.colab import drive
from pathlib import Path

drive.mount('/content/drive')

REPO_ROOT = Path('/content/drive/MyDrive/seismic-first-break-picking')
if not REPO_ROOT.exists():
    raise FileNotFoundError(f'Repo path not found: {REPO_ROOT}')

print(f'Repo root found: {REPO_ROOT}')

def print_tree(root: Path, max_depth: int = 2, prefix: str = '') -> None:
    if max_depth < 0:
        return
    entries = sorted(root.iterdir(), key=lambda p: (p.is_file(), p.name.lower()))
    for entry in entries:
        print(f'{prefix}{entry.name}/' if entry.is_dir() else f'{prefix}{entry.name}')
        if entry.is_dir() and max_depth > 0:
            print_tree(entry, max_depth=max_depth - 1, prefix=prefix + '    ')

print('Repo tree preview:')
print_tree(REPO_ROOT, max_depth=2)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Repo root found: /content/drive/MyDrive/seismic-first-break-picking
Repo tree preview:
.git/
    hooks/
        applypatch-msg.sample
        commit-msg.sample
        fsmonitor-watchman.sample
        post-update.sample
        pre-applypatch.sample
        pre-commit.sample
        pre-merge-commit.sample
        pre-push.sample
        pre-rebase.sample
        pre-receive.sample
        prepare-commit-msg.sample
        push-to-checkout.sample
        update.sample
    info/
        exclude
    logs/
        refs/
        HEAD
    objects/
        0b/
        2b/
        32/
        4d/
        50/
        51/
        57/
        5e/
        72/
        83/
        9f/
        a1/
        a6/
        ae/
        b4/
        b9/
        c3/
        c4/
        d1/
        d5/
        d6/
        db/
        e4/
        e6/
        e8/
        f4/
        f

In [4]:
import subprocess
import sys
from pathlib import Path

print('Upgrading pip...')
subprocess.run([sys.executable, '-m', 'pip', 'install', '--upgrade', 'pip'], check=True)

GENERAL_PACKAGES = [
    'h5py==3.16.0',
    'numpy==2.0.2',
    'pandas==2.2.3',
    'matplotlib==3.9.2',
    'seaborn==0.13.2',
    'scipy==1.14.1',
    'scikit-learn==1.5.2',
    'tqdm==4.67.1',
    'pyyaml==6.0.2',
    'mlflow==2.16.2',
    'optuna==4.3.0',
    'lightgbm==4.6.0',
    'xgboost==2.1.4',
    'obspy==1.4.1',
    'segmentation-models-pytorch==0.5.0',
    'albumentations==1.4.24',
]

TORCH_PACKAGES = [
    'torch==2.5.1',
    'torchvision==0.20.1',
]

print('Installing general pinned packages...')
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '--upgrade', '--no-cache-dir', *GENERAL_PACKAGES
], check=True)

print('Installing PyTorch pinned packages...')
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '--upgrade', '--no-cache-dir',
    '--index-url', 'https://download.pytorch.org/whl/cu124',
    *TORCH_PACKAGES,
], check=True)

print('Installed package versions (pip show):')
for package_name in ['torch', 'torchvision', 'h5py', 'numpy', 'mlflow']:
    print('-' * 60)
    subprocess.run([sys.executable, '-m', 'pip', 'show', package_name], check=True)

print('=' * 60)
print('CELL 2 COMPLETE')
print('Restart the Colab runtime now, then rerun from Cell 1.')
print('This avoids binary ABI conflicts after reinstalling numpy-dependent packages in-place.')
print('=' * 60)
raise SystemExit('Intentional stop after package installation. Restart runtime, then rerun from Cell 1.')


Upgrading pip...
Installing general pinned packages...
Installing PyTorch pinned packages...
torch: 2.5.1+cu124
h5py: 3.16.0
numpy: 2.0.2


ValueError: numpy.dtype size changed, may indicate binary incompatibility. Expected 96 from C header, got 88 from PyObject

In [ ]:
import torch

try:
    import torch_xla.core.xla_model as xm  # type: ignore
    TPU_AVAILABLE = True
except ImportError:
    TPU_AVAILABLE = False

if torch.cuda.is_available():
    device_type = 'CUDA'
    device_name = torch.cuda.get_device_name(0)
    props = torch.cuda.get_device_properties(0)
    vram_total_mb = props.total_memory // (1024 ** 2)
    free_bytes, total_bytes = torch.cuda.mem_get_info()
    vram_available_mb = free_bytes // (1024 ** 2)
    cuda_version = torch.version.cuda
    recommended_unet_batch = 8
    recommended_resnet_unet_batch = 4
elif TPU_AVAILABLE:
    device_type = 'TPU'
    device_name = str(xm.xla_device())
    vram_total_mb = None
    vram_available_mb = None
    cuda_version = 'N/A'
    recommended_unet_batch = 16
    recommended_resnet_unet_batch = 8
else:
    device_type = 'CPU'
    device_name = 'CPU'
    vram_total_mb = None
    vram_available_mb = None
    cuda_version = 'N/A'
    recommended_unet_batch = 1
    recommended_resnet_unet_batch = 1

DEVICE_REPORT = {
    'device_type': device_type,
    'device_name': device_name,
    'vram_total_mb': vram_total_mb,
    'vram_available_mb': vram_available_mb,
    'cuda_version': cuda_version,
    'pytorch_version': torch.__version__,
    'tpu_available': TPU_AVAILABLE,
    'recommended_batch_size_unet': recommended_unet_batch,
    'recommended_batch_size_resnet_unet': recommended_resnet_unet_batch,
}

print('=' * 60)
print('DEVICE REPORT')
print('=' * 60)
print(f"Device type     : {DEVICE_REPORT['device_type']}")
print(f"Device name     : {DEVICE_REPORT['device_name']}")
print(f"VRAM total      : {DEVICE_REPORT['vram_total_mb']} MB")
print(f"VRAM available  : {DEVICE_REPORT['vram_available_mb']} MB")
print(f"CUDA version    : {DEVICE_REPORT['cuda_version']}")
print(f"PyTorch version : {DEVICE_REPORT['pytorch_version']}")
print(f"TPU available   : {'Yes' if DEVICE_REPORT['tpu_available'] else 'No'}")
print(f"Recommended batch size (U-Net)      : {DEVICE_REPORT['recommended_batch_size_unet']}")
print(f"Recommended batch size (ResNet-UNet): {DEVICE_REPORT['recommended_batch_size_resnet_unet']}")
print('=' * 60)

if device_type == 'CPU':
    print('WARNING: CPU runtime detected. Training will be extremely slow. Switch to a GPU runtime before later phases.')


In [ ]:
import hashlib
import subprocess

RAW_DIR = REPO_ROOT / 'data' / 'raw'
RAW_DIR.mkdir(parents=True, exist_ok=True)

DATASET_SOURCES = {
    'brunswick': {
        'url': 'https://d3sakqnghgsk6x.cloudfront.net/Brunswick_3D/Brunswick_orig_1500ms_V2.hdf5.xz',
        'raw_filename': 'brunswick.hdf5.xz',
        'extracted_filename': 'brunswick.hdf5',
    },
    'halfmile': {
        'url': 'https://d3sakqnghgsk6x.cloudfront.net/Halfmile_3D/Halfmile3D_add_geom_sorted.hdf5.xz',
        'raw_filename': 'halfmile.hdf5.xz',
        'extracted_filename': 'halfmile.hdf5',
    },
    'lalor': {
        'url': 'https://d3sakqnghgsk6x.cloudfront.net/Lalor_3D/Lalor_raw_z_1500ms_norp_geom_v3.hdf5.xz',
        'raw_filename': 'lalor.hdf5.xz',
        'extracted_filename': 'lalor.hdf5',
    },
    'sudbury': {
        'url': 'https://d3sakqnghgsk6x.cloudfront.net/Sudbury_3D/preprocessed_Sudbury3D.hdf.xz',
        'raw_filename': 'sudbury.hdf.xz',
        'extracted_filename': 'sudbury.hdf5',
    },
}

def md5_for_file(path: Path, chunk_size: int = 1024 * 1024) -> str:
    digest = hashlib.md5()
    with path.open('rb') as handle:
        while True:
            chunk = handle.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)
    return digest.hexdigest()

DOWNLOAD_REPORT = {}

for asset, spec in DATASET_SOURCES.items():
    url = spec['url']
    target = RAW_DIR / spec['raw_filename']

    print()
    print(f'Downloading {asset} -> {target}')
    subprocess.run(['wget', '--continue', '--output-document', str(target), url], check=True)

    size_gb = round(target.stat().st_size / (1024 ** 3), 3)
    file_md5 = md5_for_file(target)
    DOWNLOAD_REPORT[asset] = {
        'path': str(target),
        'size_bytes': target.stat().st_size,
        'size_gb': size_gb,
        'md5': file_md5,
        'url': url,
    }
    print(f'{asset}: {size_gb} GB | md5={file_md5}')


In [ ]:
import h5py
import lzma
import shutil

EXTRACTED_DIR = REPO_ROOT / 'data' / 'extracted'
EXTRACTED_DIR.mkdir(parents=True, exist_ok=True)

EXTRACTION_REPORT = {}

def is_valid_hdf5(path: Path) -> bool:
    try:
        with h5py.File(path, 'r') as handle:
            _ = list(handle.keys())
        return True
    except Exception:
        return False

for asset, spec in DATASET_SOURCES.items():
    source = RAW_DIR / spec['raw_filename']
    target = EXTRACTED_DIR / spec['extracted_filename']

    if target.exists() and is_valid_hdf5(target):
        print(f'Skipping decompression - {target.name} already exists and opens as HDF5.')
    else:
        if target.exists():
            print(f'Removing partial or invalid file before retry: {target}')
            target.unlink()

        print(f'Decompressing {source.name} -> {target.name}')
        with lzma.open(source, 'rb') as src, target.open('wb') as dst:
            shutil.copyfileobj(src, dst)

        if not is_valid_hdf5(target):
            raise RuntimeError(f'Decompressed file is not a valid HDF5: {target}')

    size_gb = round(target.stat().st_size / (1024 ** 3), 3)
    EXTRACTION_REPORT[asset] = {
        'path': str(target),
        'size_bytes': target.stat().st_size,
        'size_gb': size_gb,
        'valid_hdf5': True,
    }
    print(f'{asset}: {size_gb} GB extracted and verified.')


In [ ]:
import h5py
import numpy as np

EXPECTED_KEYS = [
    'data_array', 'SHOTID', 'SHOT_PEG', 'SOURCE_X', 'SOURCE_Y', 'SOURCE_HT',
    'REC_PEG', 'REC_X', 'REC_Y', 'REC_HT', 'SAMP_RATE', 'COORD_SCALE',
    'HT_SCALE', 'SAMP_NUM', 'SPARE1',
]

HDF5_REPORT = {}

def summarize_constant(dataset) -> dict:
    values = dataset[()]
    unique = np.unique(values)
    return {
        'is_constant': bool(unique.size == 1),
        'unique_count': int(unique.size),
        'value': unique[0].item() if unique.size == 1 else None,
    }

for asset, spec in DATASET_SOURCES.items():
    path = EXTRACTED_DIR / spec['extracted_filename']
    print('=' * 60)
    print(f'HDF5 STRUCTURAL VERIFICATION - {asset}')
    print('=' * 60)

    with h5py.File(path, 'r') as handle:
        root_groups = list(handle.keys())
        if 'TRACE_DATA' not in handle or 'DEFAULT' not in handle['TRACE_DATA']:
            raise KeyError(f'{asset}: TRACE_DATA/DEFAULT not found')

        group = handle['TRACE_DATA']['DEFAULT']
        present_keys = list(group.keys())
        missing_keys = [key for key in EXPECTED_KEYS if key not in group]
        undocumented_keys = [key for key in present_keys if key not in EXPECTED_KEYS]

        if missing_keys:
            raise KeyError(f'{asset}: missing keys {missing_keys}')

        key_summaries = {}
        for key in EXPECTED_KEYS:
            ds = group[key]
            key_summaries[key] = {'shape': list(ds.shape), 'dtype': str(ds.dtype)}
            print(f'{key:12s}: shape={ds.shape}, dtype={ds.dtype}')

        samp_rate = summarize_constant(group['SAMP_RATE'])
        coord_scale = summarize_constant(group['COORD_SCALE'])
        ht_scale = summarize_constant(group['HT_SCALE'])
        samp_num = summarize_constant(group['SAMP_NUM'])

        spare1 = group['SPARE1'][()]
        unique_values, counts = np.unique(spare1, return_counts=True)
        labeled_mask = np.logical_and(spare1 != 0, spare1 != -1)
        labeled_traces = int(labeled_mask.sum())
        total_traces = int(spare1.shape[0])

        HDF5_REPORT[asset] = {
            'file_path': str(path),
            'file_size_bytes': path.stat().st_size,
            'root_groups': root_groups,
            'target_group_found': True,
            'missing_keys': missing_keys,
            'undocumented_keys': undocumented_keys,
            'datasets': key_summaries,
            'samp_rate': samp_rate,
            'coord_scale': coord_scale,
            'ht_scale': ht_scale,
            'samp_num': samp_num,
            'labeled_traces': labeled_traces,
            'total_traces': total_traces,
            'labeled_fraction': labeled_traces / total_traces if total_traces else None,
            'spare1_value_counts': {str(v): int(c) for v, c in zip(unique_values.tolist(), counts.tolist())},
            'status': 'PASS',
        }

        print(f'Root groups: {root_groups}')
        print(f'Undocumented keys: {undocumented_keys if undocumented_keys else "none"}')
        print(f'SAMP_RATE constant: {samp_rate["is_constant"]} | value={samp_rate["value"]}')
        print(f'COORD_SCALE constant: {coord_scale["is_constant"]} | value={coord_scale["value"]}')
        print(f'HT_SCALE constant: {ht_scale["is_constant"]} | value={ht_scale["value"]}')
        print(f'SAMP_NUM constant: {samp_num["is_constant"]} | value={samp_num["value"]}')
        print(f'Labeled traces: {labeled_traces} / {total_traces} ({labeled_traces / total_traces:.2%})')
        print('STATUS: PASS')


In [ ]:
import matplotlib.pyplot as plt

SANITY_PLOT_PATH = REPO_ROOT / 'results' / 'sanity_plots' / '00_first_traces.png'
SANITY_PLOT_PATH.parent.mkdir(parents=True, exist_ok=True)

rng = np.random.default_rng(42)
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
axes = axes.flatten()

SANITY_REPORT = {}

for ax, (asset, spec) in zip(axes, DATASET_SOURCES.items()):
    path = EXTRACTED_DIR / spec['extracted_filename']
    with h5py.File(path, 'r') as handle:
        group = handle['TRACE_DATA']['DEFAULT']
        traces = group['data_array']
        spare1 = group['SPARE1'][()]

        labeled_indices = np.where(np.logical_and(spare1 != 0, spare1 != -1))[0]
        if labeled_indices.size == 0:
            raise RuntimeError(f'No labeled traces found for {asset}')

        idx = int(rng.choice(labeled_indices))
        trace = traces[idx]
        first_break_ms = float(spare1[idx])
        samp_rate_us = group['SAMP_RATE'][idx].item()
        time_axis_ms = np.arange(trace.shape[0]) * (samp_rate_us / 1000.0)

        ax.plot(time_axis_ms, trace, linewidth=0.8)
        ax.axvline(first_break_ms, color='red', linestyle='--', linewidth=1.2)
        ax.set_title(f'{asset} | trace {idx}')
        ax.set_xlabel('Time (ms)')
        ax.set_ylabel('Amplitude')

        SANITY_REPORT[asset] = {
            'trace_index': idx,
            'first_break_ms': first_break_ms,
            'sample_rate_us': samp_rate_us,
        }

plt.tight_layout()
plt.savefig(SANITY_PLOT_PATH, dpi=200, bbox_inches='tight')
plt.show()
print(f'Saved sanity plot: {SANITY_PLOT_PATH}')


In [ ]:
import json

REPORT_PATH = REPO_ROOT / 'results' / '00_environment_report.json'
REPORT_PATH.parent.mkdir(parents=True, exist_ok=True)

full_report = {
    'repo_root': str(REPO_ROOT),
    'device_report': DEVICE_REPORT,
    'download_report': DOWNLOAD_REPORT,
    'extraction_report': EXTRACTION_REPORT,
    'hdf5_report': HDF5_REPORT,
    'sanity_report': SANITY_REPORT,
    'sanity_plot_path': str(SANITY_PLOT_PATH),
}

with REPORT_PATH.open('w', encoding='utf-8') as handle:
    json.dump(full_report, handle, indent=2)

print('=' * 60)
print('PHASE 0.5 COMPLETE')
print('All four assets verified. Environment ready for Phase 1.')
print(f'Environment report saved to: {REPORT_PATH}')
print('=' * 60)
